In [3]:
import json
import os
import re

# Define paths
vqa_dataset_path = r"d:\Workspace\Repository\thesis\research\image-retrieval-engine\data\vqa\vqa_dataset.json"
reference_path = r"d:\Workspace\Repository\thesis\research\image-retrieval-engine\data\vqa\reference.json"
metadata_path = r"d:\Workspace\Repository\thesis\research\react-agent-engine\data\plantwild\evaluation\metadata.json"
identification_path = r"d:\Workspace\Repository\thesis\research\react-agent-engine\data\langsmith\vqa_identification_examples.json"
output_path = r"d:\Workspace\Repository\thesis\research\image-retrieval-engine\data\vqa\samples.json"

# Load files
with open(vqa_dataset_path, "r") as f:
    vqa_dataset = json.load(f)

with open(reference_path, "r") as f:
    reference_data = json.load(f)

with open(metadata_path, "r") as f:
    metadata_data = json.load(f)

with open(identification_path, "r") as f:
    identification_data = json.load(f)

# Create mappings
# 1. Filename -> Image URL
filename_to_url = {item["metadata"]["filename"]: item["inputs"]["image_url"] for item in vqa_dataset}

# 2. Filename -> Pathogen Type
filename_to_pathogen = {item["metadata"]["filename"]: item["metadata"]["pathogen_type"] for item in identification_data}

# 3. Class + Status -> Management/Maintenance
class_status_to_management = {}
for item in reference_data:
    key = (item["class"], item["status"])
    if item["status"] == "healthy":
        class_status_to_management[key] = item.get("maintenance", "")
    else:
        class_status_to_management[key] = item.get("management", "")

# 4. Class -> Scientific Name extraction
class_to_scientific = {}
for item in reference_data:
    if item["status"] == "diseased":
        text = item.get("management", "")
        # Look for (Scientific Name) or "caused by bacterium Name"
        match = re.search(r"\(([A-Z][a-z]+ [a-z]+(?: pv\. [a-z]+)?)\)", text)
        if not match:
            # Try searching for patterns like "caused by fungus Venturia inaequalis"
            match = re.search(r"(?:caused by|fungus|bacterium|pathogen|virus)\s+([A-Z][a-z]+ [a-z]+(?: pv\. [a-z]+)?)", text, re.IGNORECASE)
        
        if match:
            class_to_scientific[item["class"]] = match.group(1)

# Build new dataset
new_samples = {}

for filename, item_metadata in metadata_data.items():
    if filename not in filename_to_url:
        continue
        
    cls = item_metadata["class"]
    status = item_metadata["status"]
    pathogen_type = filename_to_pathogen.get(filename, "unknown")
    
    # Determine cause
    if pathogen_type == "healthy":
        cause = "N/A - healthy plant"
    else:
        sci_name = class_to_scientific.get(cls)
        type_str = pathogen_type.capitalize()
        if sci_name:
            cause = f"{type_str} ({sci_name})"
        else:
            cause = type_str
    
    management = class_status_to_management.get((cls, status), "")
    
    new_samples[filename] = {
        "label": item_metadata["label"],
        "class": cls,
        "status": status,
        "caption": item_metadata["caption"],
        "cause": cause,
        "management": management,
        "original_path": item_metadata["original_path"],
        "path": f"images\\{filename}",
        "image_url": filename_to_url[filename]
    }

# Save the result
with open(output_path, "w") as f:
    json.dump(new_samples, f, indent=2)

print(f"Successfully created {len(new_samples)} entries in {output_path}")


Successfully created 267 entries in d:\Workspace\Repository\thesis\research\image-retrieval-engine\data\vqa\samples.json
